## Stage 2 (Round 2) — Representation Fidelity Map: layer x component x prompt, one pass

Context: `notes/research_question/04_pivot3_representation_fidelity.md` (S13.4, three rounds),
findings so far: `notes/findings/` 01-04.

**The question of this notebook:** findings 01-04 (weak/noisy/lopsided signal) were measured with
ONE readout: last token, residual stream, one prompt template. Is that weakness a
property of the model, or a property of where we stick our "electrode"? Here we attach electrodes at
EVERY location and build a map: at which location is fidelity (rho against real survey opinion distance)
highest?

Three axes varied within ONE GPU pass:

1. **Layer** — 33 residual stream points (like notebook 07, but now complete per type).
2. **Component** — residual vs **attention head** (32 layers x 32 heads = 1024 readout points,
   taken from the `o_proj` input — the SAME readout point as llm-opinions, only using
   a plain PyTorch forward hook, not PyVene) vs **FFN/MLP output** (32 points).
3. **Prompt/position** — 4 template variants per group (the construct-validity critique
   arXiv:2601.18486: one template = just one operationalization). Variant T3 ends with
   `Answer:` so the last-token readout position = "after Answer:" in llm-opinions style.

The SAME yardstick for every point: Spearman rho between the embedding distance across groups vs
`group_real_dist` (Wasserstein distance of the real survey answer distributions), computed PER TYPE
(lesson from finding 02: pooling hides the imbalance across types).

**Patching (the causal axis) is deliberately NOT here** — its design depends on this map.


## Before running: Kaggle setup

1. **Accelerator**: GPU T4 x2. **Internet: On**.
2. **Attach dataset** `opinionqa_intersectional.csv` (same as notebook 07/08).
3. If this session is left over from a crash/OOM: **RESTART SESSION** first (do not just re-run cells).
4. **WHEN DONE (IMPORTANT — do not forget this time):** download from
   `/kaggle/working/stage2_fidelity_map/`:
   - `fidelity_map_full.csv` (REQUIRED — the whole map, small)
   - `fidelity_map_top.csv` (REQUIRED — summary of the best locations)
   - `emb_resid.npz`, `emb_heads.npz`, `emb_mlp.npz` (~350MB each — download
     at least `emb_heads.npz` so follow-up head-level analysis needs no GPU)
   - all `.png` files

Time estimate: model load ~5-10 min, extraction of 676 prompts ~30-45 min, CPU analysis
~5-10 min. Total session ~1 hour.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy scikit-learn tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, spearmanr
from sklearn.metrics import pairwise_distances
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# OOM lesson #2 (notebook 08): an old crash traceback can pin GPU tensors in the kernel.
sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  WARNING: GPU {d} is not empty -> RESTART SESSION before loading the model!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError(
        "opinionqa_intersectional.csv not found. Attach the Kaggle dataset first."
    )
print("Using data from:", DATA_PATH)

N_QKEYS_FOR_WD = 300   # sample of shared questions per cell pair (same as notebook 07)
RANDOM_SEED = 42

OUT_DIR = "/kaggle/working/stage2_fidelity_map"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)


## 1. Data & intersectional cell metadata (exactly as in notebook 07)

In [ ]:
df = pd.read_csv(DATA_PATH)

def _parse_list(x):
    return ast.literal_eval(x) if isinstance(x, str) else x

df["responses"] = df["responses"].apply(_parse_list)
df["ordinal"] = df["ordinal"].apply(_parse_list)
df["options"] = df["options"].apply(_parse_list)
df["group_key"] = df["attribute"] + " :: " + df["group"]

GROUP_KEYS = sorted(df["group_key"].unique().tolist())
n_g = len(GROUP_KEYS)

group_meta = {}
for gk in GROUP_KEYS:
    attr_type, group_str = gk.split(" :: ", 1)
    v1, v2 = group_str.split(" | ", 1)
    group_meta[gk] = {"attr_type": attr_type, "v1": v1, "v2": v2}

attr_types = np.array([group_meta[gk]["attr_type"] for gk in GROUP_KEYS])
TYPES = sorted(set(attr_types.tolist()))
type_idx = {t: np.where(attr_types == t)[0] for t in TYPES}

print(f"{n_g} intersectional cells, {len(TYPES)} combination types:")
for t in TYPES:
    print(f"  {t}: {len(type_idx[t])} cells")


## 2. Real distance between cells (`group_real_dist`) — the yardstick for every readout point

In [ ]:
resp_lookup = {
    (gk, qk): (resp, ordv)
    for gk, qk, resp, ordv in zip(df["group_key"], df["qkey"], df["responses"], df["ordinal"])
}
qkeys_by_group = df.groupby("group_key")["qkey"].apply(set).to_dict()

rng = np.random.default_rng(RANDOM_SEED)
group_real_dist = np.full((n_g, n_g), np.nan)

for i in tqdm(range(n_g), desc="Real distance between cells"):
    gi = GROUP_KEYS[i]
    for j in range(i, n_g):
        if i == j:
            group_real_dist[i, j] = 0.0
            continue
        gj = GROUP_KEYS[j]
        shared = qkeys_by_group.get(gi, set()) & qkeys_by_group.get(gj, set())
        if not shared:
            continue
        shared = list(shared)
        if len(shared) > N_QKEYS_FOR_WD:
            idx = rng.choice(len(shared), size=N_QKEYS_FOR_WD, replace=False)
            shared = [shared[k] for k in idx]
        wds = []
        for qk in shared:
            respA, ordA = resp_lookup[(gi, qk)]
            respB, ordB = resp_lookup[(gj, qk)]
            if len(ordA) != len(ordB):
                continue
            wds.append(wasserstein_distance(ordA, ordB, u_weights=respA, v_weights=respB))
        if wds:
            group_real_dist[i, j] = group_real_dist[j, i] = float(np.mean(wds))

iu_all = np.triu_indices(n_g, k=1)
print("NaN pairs:", int(np.isnan(group_real_dist[iu_all]).sum()), "out of", len(iu_all[0]))


## 3. Multi-cue: 4 prompt variants per cell

Why 4: the construct-validity critique (arXiv:2601.18486) — one template is only one way of
operationalizing "group identity". If the fidelity map changes drastically across
templates, the signal is fragile (a template artifact), not a property of the model.

- **T0** third-person declarative (identical to notebook 07 — the bridge to findings 01-04)
- **T1** first-person ("I am ...")
- **T2** structured profile (bullets)
- **T3** QA format ending with a demographic answer — last token = **"after Answer:"**,
  the readout position llm-opinions validated for their first-order readout.


In [ ]:
ATTR_LABELS = {
    "RACExRELIG":       ("race", "religion"),
    "RACExPOLPARTY":    ("race", "political party affiliation"),
    "RACExPOLIDEOLOGY": ("race", "political ideology"),
    "RELIGxPOLPARTY":   ("religion", "political party affiliation"),
    "EDUCATIONxINCOME": ("highest level of education", "household income"),
    "AGExPOLPARTY":     ("age group", "political party affiliation"),
}

def make_templates(attr_type, v1, v2):
    l1, l2 = ATTR_LABELS[attr_type]
    return [
        f"This survey respondent's {l1} is {v1} and their {l2} is {v2}.",
        f"I am answering this survey. My {l1} is {v1} and my {l2} is {v2}.",
        f"Survey respondent profile:\n- {l1}: {v1}\n- {l2}: {v2}\nThis respondent is about to answer opinion questions.",
        f"Question: What is this survey respondent's {l1} and {l2}?\nAnswer: Their {l1} is {v1} and their {l2} is {v2}.",
    ]

N_TEMPLATES = 4
prompts_by_template = {t: {} for t in range(N_TEMPLATES)}
for gk in GROUP_KEYS:
    m = group_meta[gk]
    for t, p in enumerate(make_templates(m["attr_type"], m["v1"], m["v2"])):
        prompts_by_template[t][gk] = p

print(f"Total prompts: {N_TEMPLATES} x {n_g} = {N_TEMPLATES * n_g}\n")
gk0 = GROUP_KEYS[0]
for t in range(N_TEMPLATES):
    print(f"--- T{t} ---")
    print(prompts_by_template[t][gk0])
    print()


## 4. Load model & extract the 3 components at once

A single forward pass photographs, at the **last token**:

- **residual**: `hidden_states` (33 x 4096) — 1 initial snapshot + 32 post-block
- **head**: `o_proj` input per layer (concat of 32 heads x 128 dims, BEFORE being mixed by o_proj) —
  the same readout point as llm-opinions (`self_attn.o_proj.input`), reshaped to [head, 128]
- **mlp**: FFN block output per layer (32 x 4096)


In [ ]:
print(f"Loading tokenizer & model: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
NUM_HEADS = model.config.num_attention_heads
HIDDEN = model.config.hidden_size
HEAD_DIM = HIDDEN // NUM_HEADS
print(f"{NUM_LAYERS} layers, {NUM_HEADS} heads x {HEAD_DIM} dims, hidden {HIDDEN}")
if torch.cuda.is_available():
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d} after load: {free/1e9:.1f} GB free")


In [ ]:
_capture = {}

def _oproj_prehook(layer_idx):
    def fn(module, args):
        _capture[("head", layer_idx)] = args[0][0, -1, :].detach().float().cpu()
    return fn

def _mlp_hook(layer_idx):
    def fn(module, args, output):
        _capture[("mlp", layer_idx)] = output[0, -1, :].detach().float().cpu()
    return fn

handles = []
for li, layer in enumerate(model.model.layers):
    handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(li)))
    handles.append(layer.mlp.register_forward_hook(_mlp_hook(li)))
print(f"{len(handles)} hooks attached.")

@torch.no_grad()
def extract_all(prompt):
    _capture.clear()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model(**inputs, output_hidden_states=True)
    resid = torch.stack(out.hidden_states, dim=0)[:, 0, -1, :].float().cpu().numpy()
    heads = np.stack([_capture[("head", li)].numpy().reshape(NUM_HEADS, HEAD_DIM)
                      for li in range(NUM_LAYERS)])
    mlp = np.stack([_capture[("mlp", li)].numpy() for li in range(NUM_LAYERS)])
    return resid, heads, mlp

resid_all = np.zeros((N_TEMPLATES, n_g, NUM_LAYERS + 1, HIDDEN), dtype=np.float16)
heads_all = np.zeros((N_TEMPLATES, n_g, NUM_LAYERS, NUM_HEADS, HEAD_DIM), dtype=np.float16)
mlp_all   = np.zeros((N_TEMPLATES, n_g, NUM_LAYERS, HIDDEN), dtype=np.float16)

for t in range(N_TEMPLATES):
    for i, gk in enumerate(tqdm(GROUP_KEYS, desc=f"Extraction T{t}")):
        r, h, m = extract_all(prompts_by_template[t][gk])
        resid_all[t, i], heads_all[t, i], mlp_all[t, i] = r, h, m
    torch.cuda.empty_cache()

for h_ in handles:
    h_.remove()
print("Extraction done, hooks removed.")

gkeys_arr = np.array(GROUP_KEYS, dtype=object)
np.savez_compressed(os.path.join(OUT_DIR, "emb_resid.npz"), emb=resid_all, group_keys=gkeys_arr)
np.savez_compressed(os.path.join(OUT_DIR, "emb_heads.npz"), emb=heads_all, group_keys=gkeys_arr)
np.savez_compressed(os.path.join(OUT_DIR, "emb_mlp.npz"),   emb=mlp_all,   group_keys=gkeys_arr)
np.save(os.path.join(OUT_DIR, "group_real_dist.npy"), group_real_dist)
for f in ["emb_resid.npz", "emb_heads.npz", "emb_mlp.npz"]:
    sz = os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6
    print(f"  {f}: {sz:.0f} MB")


## 5. Fidelity map: rho per (component x location x template x type)

One function, used at every readout point. Rho is computed **within-type** (all cell pairs
in 1 combination type) — lesson from finding 02, pooled is misleading. Besides per-template, there is a
**Tmean** variant (embeddings averaged across the 4 templates first) = the multi-cue readout.


In [ ]:
def within_type_rho(emb_2d, subset_idx):
    sub = emb_2d[subset_idx].astype(np.float32)
    rep = pairwise_distances(sub, metric="cosine")
    real = group_real_dist[np.ix_(subset_idx, subset_idx)]
    iu = np.triu_indices(len(subset_idx), k=1)
    a, b = real[iu], rep[iu]
    ok = ~np.isnan(a) & ~np.isnan(b)
    if ok.sum() < 5:
        return np.nan
    return float(spearmanr(a[ok], b[ok])[0])

import warnings
warnings.filterwarnings("ignore")  # spearmanr ConstantInputWarning on small subsets

TEMPLATE_TAGS = [f"T{t}" for t in range(N_TEMPLATES)] + ["Tmean"]

def emb_at(source, t_tag, loc):
    """source: 'resid'|'mlp'|'head'; loc: layer for resid/mlp, (layer, head) for head."""
    if source == "resid":
        arr = resid_all[:, :, loc, :]
    elif source == "mlp":
        arr = mlp_all[:, :, loc, :]
    else:
        li, hi = loc
        arr = heads_all[:, :, li, hi, :]
    if t_tag == "Tmean":
        return arr.astype(np.float32).mean(axis=0)
    return arr[int(t_tag[1])]

rows = []

# residual: 33 locations
for L in tqdm(range(NUM_LAYERS + 1), desc="residual map"):
    for t_tag in TEMPLATE_TAGS:
        e = emb_at("resid", t_tag, L)
        for ty in TYPES:
            rows.append(dict(component="resid", layer=L, head=-1, template=t_tag,
                             attr_type=ty, rho=within_type_rho(e, type_idx[ty])))

# mlp: 32 locations
for L in tqdm(range(NUM_LAYERS), desc="mlp map"):
    for t_tag in TEMPLATE_TAGS:
        e = emb_at("mlp", t_tag, L)
        for ty in TYPES:
            rows.append(dict(component="mlp", layer=L, head=-1, template=t_tag,
                             attr_type=ty, rho=within_type_rho(e, type_idx[ty])))

fmap_rm = pd.DataFrame(rows)
print(fmap_rm.shape)


In [ ]:
# head: 32x32 = 1024 locations. Per-template + Tmean x 6 types = ~30k correlations (CPU, a few minutes).
rows_h = []
for L in tqdm(range(NUM_LAYERS), desc="head map"):
    for H in range(NUM_HEADS):
        for t_tag in TEMPLATE_TAGS:
            e = emb_at("head", t_tag, (L, H))
            for ty in TYPES:
                rows_h.append(dict(component="head", layer=L, head=H, template=t_tag,
                                   attr_type=ty, rho=within_type_rho(e, type_idx[ty])))

fmap_head = pd.DataFrame(rows_h)
fmap = pd.concat([fmap_rm, fmap_head], ignore_index=True)
fmap.to_csv(os.path.join(OUT_DIR, "fidelity_map_full.csv"), index=False)
print("Full map:", fmap.shape, "-> fidelity_map_full.csv")


## 6. Reading the map

Comparison baseline (the naive readout, findings 01-02): residual T0, per type —
AGExPOLPARTY ~0.50, EDUCATIONxINCOME ~0.49, RACExRELIG ~0.07.
The question: is there a location that BEATS it by a wide margin — especially for the weak types?


In [ ]:
# 6a. Top-20 locations per type (using Tmean so it is not a 1-template artifact)
tm = fmap[fmap["template"] == "Tmean"].copy()
top_rows = []
print("=" * 80)
for ty in TYPES:
    sub = tm[tm["attr_type"] == ty].sort_values("rho", ascending=False)
    naive = tm[(tm["attr_type"] == ty) & (tm["component"] == "resid")]["rho"].max()
    best = sub.iloc[0]
    print(f"\n{ty}  (best residual: {naive:+.3f})")
    print(sub.head(10)[["component", "layer", "head", "rho"]].to_string(index=False))
    top_rows.append(sub.head(20))
pd.concat(top_rows).to_csv(os.path.join(OUT_DIR, "fidelity_map_top.csv"), index=False)


In [ ]:
# 6b. Residual & mlp curves per layer, per type (Tmean)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for comp, ax in zip(["resid", "mlp"], axes):
    sub = tm[tm["component"] == comp]
    for ty in TYPES:
        s = sub[sub["attr_type"] == ty].sort_values("layer")
        ax.plot(s["layer"], s["rho"], marker=".", label=ty)
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_title(f"{comp} — rho per layer (Tmean)")
    ax.set_xlabel("layer")
axes[0].set_ylabel("Spearman rho vs real survey distance")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "map_per_layer_resid_mlp.png"), dpi=150)
plt.show()


In [ ]:
# 6c. 32x32 head heatmap per type (Tmean)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
hm_sub = tm[tm["component"] == "head"]
for ax, ty in zip(axes.flat, TYPES):
    grid = np.full((NUM_LAYERS, NUM_HEADS), np.nan)
    s = hm_sub[hm_sub["attr_type"] == ty]
    grid[s["layer"].values, s["head"].values] = s["rho"].values
    im = ax.imshow(grid, aspect="auto", cmap="RdBu_r", vmin=-0.6, vmax=0.6)
    ax.set_title(ty, fontsize=10)
    ax.set_xlabel("head")
    ax.set_ylabel("layer")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label="rho (Tmean)")
plt.savefig(os.path.join(OUT_DIR, "fidelity_map_head_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 6d. Template sensitivity: at each type's best location, how much does rho wobble across T0-T3?
print("Best location (Tmean) per type + rho per template — wobble = fragile signal:\n")
sens_rows = []
for ty in TYPES:
    sub = tm[tm["attr_type"] == ty].sort_values("rho", ascending=False).iloc[0]
    comp, L, H = sub["component"], int(sub["layer"]), int(sub["head"])
    selected = fmap[(fmap["component"] == comp) & (fmap["layer"] == L) &
               (fmap["head"] == H) & (fmap["attr_type"] == ty)]
    per_t = selected.set_index("template")["rho"]
    spread = float(per_t[[f"T{t}" for t in range(N_TEMPLATES)]].std())
    print(f"{ty:20s} {comp:5s} L{L:2d} H{H:3d} | " +
          " ".join(f"{tag}={per_t[tag]:+.2f}" for tag in TEMPLATE_TAGS) +
          f" | std across T: {spread:.3f}")
    sens_rows.append(dict(attr_type=ty, component=comp, layer=L, head=H,
                          std_across_templates=spread, **{k: float(v) for k, v in per_t.items()}))
pd.DataFrame(sens_rows).to_csv(os.path.join(OUT_DIR, "template_sensitivity.csv"), index=False)


## How to read the results & checklist

**Three questions this map answers:**

1. **Is there a location far more faithful than the naive readout?** See 6a: compare the best
   rho per type vs the residual baseline. If some head/layer breaks far above
   0.50 (especially for RACExRELIG, which was 0.07) -> Round 2 goes on focused there,
   and the positive branch of Round 3 opens up.
2. **Is the signal a model property or a template artifact?** See 6d: a best location whose
   across-template std is large = fragile, do not trust it.
3. **What if every location stays weak?** That is a finding too (the negative branch of Round 3):
   demographic identity in an LLM is shallow — readable per group, not organized across
   groups — at EVERY reasonable readout point.

**Download checklist (DO NOT skip):**
`fidelity_map_full.csv`, `fidelity_map_top.csv`, `template_sensitivity.csv`,
both `.png` files, `group_real_dist.npy`, and at least `emb_heads.npz`.

Run results -> paste the output of cells 6a & 6d into the discussion; findings go into `notes/findings/05_...`.
